# Exercise: the detective game

**Duration** ~40 min &nbsp;·&nbsp; **Session** Day 2, morning

Two very different kinds of object, pooled into one table with the labels
stripped off. Your job is to tell them apart **using shape alone** — no
intensities, and no peeking at which file a row came from until the end.

This is the shape of a real problem: you have a mixed population and want to
count how many of each type are present.

**Data**: `data/bbbc010/` (*C. elegans* worms) and `data/bbbc030/` (CHO cells).

In [ ]:
import numpy as np
import pandas as pd
import tifffile
import matplotlib.pyplot as plt

from skimage.measure import regionprops_table

from course import DATA, show

## Task 1 — look at what you are dealing with

Display one worm image and one cell image side by side, with their annotations.

<details>
<summary>Hint</summary>

`data/bbbc010/images/A22_gfp.tif` and `data/bbbc030/images/cho01_dic.tif`;
the annotations are in the matching `gt/` folders, named `*_worm_labels.tif`
and `*_cell_labels.tif`.
</details>

In [ ]:
from skimage.color import label2rgb

# --- your turn ---
worm_image = ...    # TODO
worm_labels = ...   # TODO
cell_image = ...    # TODO
cell_labels = ...   # TODO

... display all four side by side ...

**Your answer:** before measuring anything, write down two or three shape
properties you expect to separate them. You will check these against the data in
task 4. *(edit this cell)*

## Task 2 — build the pooled table

Measure every annotated object in **both** datasets and put them in one table.
Keep a `kind` column recording which dataset each row came from — that is your
answer key, and you must not use it as a feature.

<details>
<summary>Hint 1 — which properties?</summary>

Shape only: `area`, `perimeter`, `eccentricity`, `solidity`, `extent`,
`axis_major_length`, `axis_minor_length`. No `intensity_*`.
</details>

<details>
<summary>Hint 2 — looping over the files</summary>

`sorted((DATA / "bbbc010" / "gt").glob("*_worm_labels.tif"))` gives the worm
annotations; the cells are `*_cell_labels.tif` under `bbbc030`.
</details>

In [ ]:
SHAPE_FEATURES = ("area", "perimeter", "eccentricity", "solidity", "extent",
                  "axis_major_length", "axis_minor_length")

# --- your turn ---
def measure_all(folder, pattern, kind):
    ...   # TODO: measure every label image matching `pattern`

objects = pd.concat([...], ignore_index=True)   # TODO: both datasets

print(objects.groupby("kind").size().to_string())
objects.head()

## Task 3 — add a ratio of your own

`area` and the axis lengths are all in pixels, so they depend on magnification.
Add an **aspect ratio** column — major axis divided by minor axis — which does
not.

<details>
<summary>Hint</summary>

Guard against a zero minor axis: `.clip(lower=1)` on the denominator.
</details>

In [ ]:
# --- your turn ---
objects["aspect"] = ...   # TODO

objects.groupby("kind")[["area", "eccentricity", "solidity", "extent", "aspect"]].median().round(2)

## Task 4 — which feature separates them?

Plot the distribution of each candidate feature for the two kinds, overlaid, and
decide which separates them best.

<details>
<summary>Hint</summary>

For each feature, `plt.hist` twice on the same axes with `alpha=0.6` and a
`label`, one call per kind. A shared `bins` argument makes them comparable.
</details>

In [ ]:
candidates = ["area", "eccentricity", "solidity", "extent", "aspect"]

# --- your turn ---
fig, axes = plt.subplots(1, len(candidates), figsize=(20, 3.2))
for ax, column in zip(axes, candidates):
    ...   # TODO: overlay a histogram per kind
plt.tight_layout(); plt.show()

**Your answer:** which feature separates the two best, and does it match what you
predicted in task 1? *(edit this cell)*

## Task 5 — put a number on it

An overlaid histogram is a judgement call. Score each feature instead: for every
possible cut-off, what fraction of objects would be classified correctly? Report
the best.

<details>
<summary>Hint 1 — the idea</summary>

For a candidate cut-off `v`, one rule is "worm if feature > v". Its accuracy is
the mean of *(fraction of worms above v)* and *(fraction of cells at or below
v)*. The opposite rule is worth checking too, since some features are larger for
cells.
</details>

<details>
<summary>Hint 2 — which cut-offs to try</summary>

Every distinct value the feature takes: `np.unique(objects[column])`.
</details>

In [ ]:
def best_split(table, column):
    """Best single-threshold accuracy for `column`, and where it falls."""
    worms = table.loc[table.kind == "worm", column].to_numpy()
    cells = table.loc[table.kind == "cell", column].to_numpy()

    # --- your turn ---
    best_accuracy, best_value = 0.0, None
    for value in ...:            # TODO: every distinct value
        above = ...              # TODO: accuracy of "worm if > value"
        below = ...              # TODO: accuracy of "worm if <= value"
        ...                      # TODO: keep the best
    return best_accuracy, best_value


print(f"{'feature':20s} {'accuracy':>9s} {'cut-off':>10s}")
print("-" * 41)
for column in candidates:
    accuracy, value = best_split(objects, column)
    print(f"{column:20s} {accuracy:9.3f} {value:10.2f}")

**Your answer:** `area` is the feature most people reach for first. Where does it
rank? Why do the shape ratios do better? *(edit this cell)*

## Task 6 — the honest check

Apply your best single rule and look at what it gets wrong. Print how many
objects of each kind are misclassified, and display a couple of the mistakes.

<details>
<summary>Hint</summary>

Build a boolean prediction column from your cut-off, then
`pd.crosstab(objects.kind, prediction)`.
</details>

In [ ]:
# --- your turn ---
accuracy, cutoff = best_split(objects, "extent")
predicted = ...    # TODO: apply the rule

print(pd.crosstab(objects.kind, predicted, rownames=["actual"], colnames=["predicted"]))

**Your answer:** are the mistakes evenly split between the two kinds, or does the
rule fail mostly in one direction? What would you look at in the images to
understand why? *(edit this cell)*

```{note}
A single threshold on a single feature got you most of the way here, and it is
worth noticing how far that goes. Combining features — a small decision tree, or
the logistic regression you may have met elsewhere — would do a little better,
and is the natural next step. But the honest summary is that **choosing the right
feature mattered far more than choosing a clever classifier.**
```

## If you finish early

Every object here came from a perfect annotation. Repeat the exercise using
segmentations *you* produce — threshold and label the worm and cell images
yourself, then measure those.

The features will be noisier and the separation worse. That gap between "measured
from ground truth" and "measured from my pipeline" is the thing that decides
whether an analysis works in practice.